# PACE GPT-OSS Chat

This notebook shows how to run `openai/gpt-oss-20b` with PACE.

> GPT-OSS should be treated like a chat model, not like a plain raw-completion model.

## GPT-OSS Usage Notes

- GPT-OSS is trained for the harmony/chat format and should be used with the chat template. Raw prompts may produce off-format outputs, so this example applies the tokenizer chat template explicitly. See the [model card](https://huggingface.co/openai/gpt-oss-20b#transformers).
- Raw completion prompts often lead to internal analysis text or unrelated assistant-style continuations.
- GPT-OSS commonly emits both an `analysis` channel and a `final` channel.
- For user-facing output, decode the generated tokens and then extract only the text from the `final` channel.
- A lightly seeded assistant message often produces cleaner final output, but this notebook leaves the assistant unseeded so you can observe some thinking tokens in the raw stream.

## Runtime Notes

- Backend/cache has to be: `SLAB_POOL` + `SLAB`
- Check `docs/PerformanceGuide.md` for performance tips.
- Avoid stop strings like `"\n\n"` for GPT-OSS chat runs. They can stop generation inside the `analysis` channel before the model reaches `final`.
- This notebook uses deterministic generation with `do_sample=False`.


In [ ]:
import re

import torch
from transformers import AutoTokenizer, TextStreamer

import pace  # noqa: F401
from pace.llm import (
    LLMModel,
    SamplingConfig,
    KVCacheType,
    LLMOperatorType,
    OperatorConfig,
)
from pace.llm.attention import AttentionBackendType
from pace.utils.logging import suppress_logging

FINAL_MARKER = "<|channel|>final<|message|>"


## Small Note on Prompting and Cleanup

- `build_chat_prompt(...)` formats the prompt with the GPT-OSS chat template.
- `extract_final_text(...)` removes channel markers and keeps only the final answer.
- The raw stream is still shown directly so you can observe the model output as it is generated.

In [ ]:
def build_chat_prompt(tokenizer, user_prompt, assistant_seed=None):
    """Format the prompt with GPT-OSS chat tokens instead of using raw text."""
    if hasattr(tokenizer, "apply_chat_template") and getattr(tokenizer, "chat_template", None):
        if assistant_seed is not None:
            return tokenizer.apply_chat_template(
                [
                    {"role": "user", "content": user_prompt},
                    {"role": "assistant", "content": assistant_seed},
                ],
                tokenize=False,
                add_generation_prompt=False,
                continue_final_message=True,
            )
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": user_prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
    return user_prompt


def extract_final_text(tokenizer, token_ids):
    """Keep only the final user-facing answer from GPT-OSS output."""
    raw = tokenizer.decode(token_ids, skip_special_tokens=False)
    if FINAL_MARKER in raw:
        raw = raw.split(FINAL_MARKER, 1)[1]
    if tokenizer.eos_token:
        raw = raw.split(tokenizer.eos_token)[0]
    raw = raw.replace("<|return|>", "")
    raw = re.sub(r"<\|.*?\|>", "", raw)
    return raw.strip()

In [ ]:
model_name = "openai/gpt-oss-20b"
torch_dtype = torch.bfloat16

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

opconfig = OperatorConfig(**{LLMOperatorType.Attention: AttentionBackendType.SLAB})

## Model Load Parameters

- Load the model with `dtype=torch.bfloat16`.
- This notebook uses `KVCacheType.SLAB_POOL`.
- Attention backend is set through `OperatorConfig` with `AttentionBackendType.SLAB`.
- `suppress_logging()` is used during model load to keep notebook output cleaner.


In [ ]:
with suppress_logging():
    pace_model = LLMModel(
        model_name,
        dtype=torch_dtype,
        kv_cache_type=KVCacheType.SLAB_POOL,
        opconfig=opconfig,
    )

## Creative Task

This example uses a creative writing prompt rather than summarization.

Why this prompt shape:

- Deterministic generation is enabled with `do_sample=False`.
- The prompt asks for a structured answer.
- The assistant is left unseeded so the raw stream can show some `analysis` tokens before `final`.
- The generated text is still postprocessed to keep only the `final` channel after streaming.


In [ ]:
user_prompt = """You are helping a game writer.
Create a compact mission brief for a near-future sci-fi stealth game.

Requirements:
- 1 title
- 3 mission objectives
- 1 unexpected twist
- 1 closing line of radio chatter
- 120 to 180 words
- vivid but clear language
- think briefly, then provide the final answer
"""

In [ ]:
assistant_seed = None
chat_prompt = build_chat_prompt(tokenizer, user_prompt, assistant_seed=assistant_seed)
input_encoded = tokenizer.batch_encode_plus([chat_prompt], return_tensors="pt", padding="longest")

## Streaming Raw GPT-OSS Output

The notebook uses the normal `TextStreamer` so you can watch the raw GPT-OSS output as it is generated.

That means you may see:

- the `analysis` or thinking portion first
- the `final` answer after that
- the raw channel markers in the stream

`suppress_logging()` is used around model loading and generation so PACE logs do not clutter the streamed tokens. The next cell still extracts just the final answer for a clean result.

In [ ]:
sampling_config = SamplingConfig(
    max_new_tokens=750,
    do_sample=False,
    stop_strings=["<|return|>"],
)

text_streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=False)

In [ ]:
with suppress_logging():
    outputs = pace_model.generate(input_encoded, sampling_config, text_streamer)

generated_ids = outputs.output_token_ids[0, input_encoded.input_ids.shape[1]:].tolist()


In [ ]:
final_text = extract_final_text(tokenizer, generated_ids)
print(final_text)